# Qwen2-VL-2B 纯 OCR 微调
**运行环境**: Google Colab (免费 T4 GPU)  
**耗时**: ~15 分钟  
**输出**: LoRA 微调后的模型

## 第 1 步：上传训练数据
运行下面单元格后会弹出文件选择框：
- 选择你的 `data/ocr` 整个文件夹打包的 zip
- 或者分别上传 `labels.jsonl` 和 `images/` 文件夹

In [ ]:
from google.colab import files
import zipfile, os

# 方式 1：上传 zip 包（推荐）
uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('data/ocr/')
        print(f'✅ 解压 {fname} 完成')

# 验证数据
if os.path.exists('data/ocr/labels.jsonl'):
    with open('data/ocr/labels.jsonl') as f:
        n = sum(1 for _ in f)
    imgs = len(os.listdir('data/ocr/images/'))
    print(f'✅ {n} 条标注, {imgs} 张图片')
else:
    print('⚠️ 未找到 labels.jsonl，请检查上传')

## 第 2 步：安装依赖

In [ ]:
!pip install -q transformers peft datasets pillow accelerate

## 第 3 步：加载模型 + 配置 LoRA

In [ ]:
import torch, json, os
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model, TaskType

# ===== 配置 =====
MODEL_ID  = "Qwen/Qwen2-VL-2B-Instruct"
LORA_R    = 16

# 下载原始模型（约 4GB，首次需要几分钟）
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)

# 冻结视觉编码器
for p in model.model.visual.parameters():
    p.requires_grad = False

# LoRA 只微调注意力层
lora = LoraConfig(
    r=LORA_R, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1, bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 第 4 步：加载数据

In [ ]:
from torch.utils.data import Dataset, Subset
from PIL import Image

class OCRDataset(Dataset):
    def __init__(self, data_path):
        with open(data_path) as f:
            self.data = [json.loads(line) for line in f if line.strip()]
        self.base = os.path.dirname(data_path)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        img = Image.open(os.path.join(self.base, item["image"])).convert("RGB")
        return {"image": img, "text": item["text"]}

dataset = OCRDataset("data/ocr/labels.jsonl")
split = int(len(dataset) * 0.8)
train_data = Subset(dataset, range(split))
val_data   = Subset(dataset, range(split, len(dataset)))
print(f"训练集: {len(train_data)}, 验证集: {len(val_data)}")

## 第 5 步：预处理 + 训练

In [ ]:
from transformers import Trainer, TrainingArguments

processor = AutoProcessor.from_pretrained(MODEL_ID)

def collate_fn(examples):
    texts, images = [], []
    for item in examples:
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": item["image"]},
                {"type": "text", "text": "识别图片中的所有文字，只输出文字内容，不要加任何解释。"}
            ]},
            {"role": "assistant", "content": item["text"]}
        ]
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
        images.append(item["image"])

    inputs = processor(text=texts, images=images,
                       return_tensors="pt", padding=True)
    inputs["labels"] = inputs["input_ids"].clone()
    return inputs

# 训练
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./checkpoints",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        report_to="none",
        remove_unused_columns=False,
    ),
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=collate_fn,
)
trainer.train()

## 第 6 步：合并 LoRA + 打包下载

In [ ]:
from google.colab import files

# 合并 LoRA 权重
merged = model.merge_and_unload()
merged.save_pretrained("./Qwen2-VL-2B-OCR")
processor.save_pretrained("./Qwen2-VL-2B-OCR")
print("✅ 模型保存到 ./Qwen2-VL-2B-OCR")

# 打包下载
!zip -r Qwen2-VL-2B-OCR.zip Qwen2-VL-2B-OCR/
files.download("Qwen2-VL-2B-OCR.zip")

## 接下来
下载 `Qwen2-VL-2B-OCR.zip` 后，解压到服务器的 `model/` 目录，然后用和原来一样的 `export_qwen2vl_llm.py` 转换即可。